In [ ]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import re

import pdfplumber

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert




In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'ZA NCR' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)

writer = ExcelWriter(filename, engine='openpyxl')

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)

Running ZA NCR Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------



#Starting Chrome driver, set to download files in tempfolder
#Try to download the insecure file in 



chrome_options = Options()
chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--allow-running-insecure-content")  # Allow insecure content

chrome_options.add_experimental_option("prefs", {
    "download.default_directory": tempfolder,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
})

driver = webdriver.Chrome(options=chrome_options)
driver.maximize_window()

In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict



In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = { #'ZA NCR 1': 'https://www.ncr.org.za/register_of_registrants/registered_cp.php',
            'ZA NCR 2': 'https://www.ncr.org.za/register_of_registrants/registered_cb1.php',
            'ZA NCR 3': 'https://www.ncr.org.za/register_of_registrants/registered_pda.php',

            }

Typology ={

            'ZA NCR 1': 'Registered Credit Providers',
            'ZA NCR 2': 'Registered Credit Bureaus',
            'ZA NCR 3': 'Registered Payment Distribution Agency',


}


sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
         'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

In [6]:



for reg in regdict:

    print(f'Working with list {reg}')

    driver.get(regdict[reg])

    sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')

    sleep(2)

    try:
        totalPage = soup.find('div', class_='Zebra_Pagination').find('ul').find_all('li')[-2].text.strip()
    except:
        totalPage = 1


    for page in range(1, int(totalPage)+1):
        print(f'Working with page {page} of {totalPage} for {reg}')
        driver.get(f"{regdict[reg]}?page={page}")
        sleep(2)
        inner_soup = BeautifulSoup(driver.page_source, 'html.parser')
        sleep(2)
        table = inner_soup.find('div',id='output')
        rows = table.find('tbody').find_all('tr')

        for row in rows:
            td = row.find('td')
            cols = td.find_all('div', class_='col-1-2')
            # If there are no columns, skip to the next row
            if not cols:
                continue
            # Extracting columns from the row       
            else:
                name = cols[0] 
                number = cols[1]

            companyInfo = name.find('tbody')
            RegisterInfo = number.find('tbody')

            for index, tr in enumerate(companyInfo.find_all('tr')):
                tds = tr.find_all('td', style=True)
                if tds:
                    sqldict['ListProcessDate'].append(processdate)   
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])

                    if index == 0:
                        fullName = tds[-1].text
                        print(fullName)
                        sqldict['Name'].append(fullName)
                    if index == 1:
                        tradingName = tds[-1].text
                        print(tradingName)
                        sqldict['Name - Mother Company'].append(tradingName)
                    if index == 2:
                        Phone = tds[-1].text
                        #print(Phone)
                        sqldict['Phone'].append(Phone)
                    if index == 3:
                        Fax = tds[-1].text
                        #print(Fax)
                        sqldict['Fax'].append(Fax)
                    if index == 4:
                        RegisDate = tds[-1].text
                        sqldict['RegulationDate'].append(RegisDate)
            for index,tr in enumerate(RegisterInfo.find_all('tr')):
                tds = tr.find_all('td', style=True)
                if tds:
                    if index == 0:
                        NCR_no = tds[-1].text
                        #print(NCR_no)
                        sqldict['InternalID_1'].append(str(NCR_no))
                        sqldict['InternalID_1_type'].append('NCR Registration No.')
                    elif index == 1:
                        Legal_no = tds[-1].text
                        #print(Legal_no)
                        sqldict['InternalID_2'].append(str(Legal_no))
                        sqldict['InternalID_2_type'].append('Legal Registration No.')
                    elif index == 2:
                        Address = tds[-1].text
                        zip = Address.split(' ')[-1]
                        #print(Address)
                        #print(zip)
                        sqldict['Address_1'].append(Address)
                        sqldict['Zip'].append(zip)
                    elif index == 3:
                        Town = tds[-1].text
                        #print(Town)
                        sqldict['City'].append(Town)


            sqldict = bourange_same_length_array(sqldict)



Working with list ZA NCR 2
Working with page 1 of 6 for ZA NCR 2
CrossCheck Information Bureau (Pty) Ltd

Consumer Profile Bureau (Pty) Ltd
CPB
TransUnion Credit Bureau (Pty) Ltd

Xpert Decision Systems (Pty) Ltd
XDS  
TPN Group (Pty)Ltd
TPN Credit Bureau
Managed Integrity Evaluation (Pty) Ltd

Experian South Africa (Pty) Ltd

Inoxico (Pty) Ltd 

CreditWatch (Pty) Ltd
Medical Credit Watch
Southern African Fraud Prevention Service Npc

Working with page 2 of 6 for ZA NCR 2
VeriCred Credit Bureau (Pty) Ltd
VCCB
Clearscore (Pty) Ltd

Kudough Credit Solutions (Pty) Ltd

Cred-IT-Data Online Holdings (Pty) Ltd

Maris IT Development (Pty) Ltd

Omnisol Information Technology (Pty) Ltd

Lexisnexis Risk Management (Pty) Ltd

Smart Information Bureau (Pty) Ltd

Searchworks 360 (Pty) Ltd

Horizon Informatics Systems Solution (Pty) Ltd

Working with page 3 of 6 for ZA NCR 2
IDR South Africa (Pty) Ltd
V-Report 
Accountability Group (Pty) Ltd

Payprop Capital (Pty) Ltd

iFacts (Pty) Ltd

Zoia Consult

In [7]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------


os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df  = df.drop_duplicates()

df = df[df['Name'].str.strip() != '']

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)
     

C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_34304\2549480172.py:14: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [ ]:
os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)
df = df[df['Name'].str.strip() != '']

In [ ]:
df

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,CrossCheck Information Bureau (Pty) Ltd,NCRCB1,NCR Registration No.,1997/015143/07,Legal Registration No.,...,,,,,,,,,,
7,,,,,,Consumer Profile Bureau (Pty) Ltd,NCRCB2,NCR Registration No.,1981/007624/07,Legal Registration No.,...,,,CPB,,,,,,,
14,,,,,,TransUnion Credit Bureau (Pty) Ltd,NCRCB4,NCR Registration No.,2004/007773/07,Legal Registration No.,...,,,,,,,,,,
21,,,,,,Xpert Decision Systems (Pty) Ltd,NCRCB5,NCR Registration No.,2002/022938/07,Legal Registration No.,...,,,XDS,,,,,,,
28,,,,,,TPN Group (Pty)Ltd,NCRCB8,NCR Registration No.,2002/032126/07,Legal Registration No.,...,,,TPN Credit Bureau,,,,,,,
35,,,,,,Managed Integrity Evaluation (Pty) Ltd,NCRCB11,NCR Registration No.,2003/016541/07,Legal Registration No.,...,,,,,,,,,,
42,,,,,,Experian South Africa (Pty) Ltd,NCRCB16,NCR Registration No.,2006/010440/07,Legal Registration No.,...,,,,,,,,,,
49,,,,,,Inoxico (Pty) Ltd,NCRCB17,NCR Registration No.,2006/034939/07,Legal Registration No.,...,,,,,,,,,,
56,,,,,,CreditWatch (Pty) Ltd,NCRCB18,NCR Registration No.,2011/108310/07,Legal Registration No.,...,,,Medical Credit Watch,,,,,,,
63,,,,,,Southern African Fraud Prevention Service Npc,NCRCB20,NCR Registration No.,2000/020784/08,Legal Registration No.,...,,,,,,,,,,
